# 📊 Data Exploration — DecentraID Anomaly Detection

This notebook explores the synthetic access event data used for training our anomaly detection models.

## Dataset Overview
- **Source**: Synthetic access events from DecentraID platform
- **Users**: 200 simulated users
- **Events**: 20,000 total (100 per user)
- **Anomaly Ratio**: ~10%
- **Features**: 15-dimensional feature vectors

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Libraries loaded successfully!')

## 1. Load the Data

In [ ]:
# Load synthetic access data
data_path = Path('../data/synthetic_access_data.csv')
df = pd.read_csv(data_path)

print(f'Dataset shape: {df.shape}')
print(f'\nColumn types:\n{df.dtypes}')
print(f'\nFirst 5 rows:')
df.head()

## 2. Basic Statistics

In [ ]:
# Dataset summary
print('=== Dataset Summary ===')
print(f'Total events: {len(df)}')
print(f'Unique users: {df["user_id"].nunique()}')
print(f'Unique resources: {df["resource"].nunique()}')
print(f'Unique actions: {df["action"].nunique()}')
print(f'Unique IPs: {df["ip_address"].nunique()}')

# Anomaly distribution
if 'is_anomaly' in df.columns:
    anomaly_counts = df['is_anomaly'].value_counts()
    print(f'\nAnomaly distribution:')
    print(f'  Normal: {anomaly_counts.get(False, 0)} ({anomaly_counts.get(False, 0)/len(df)*100:.1f}%)')
    print(f'  Anomaly: {anomaly_counts.get(True, 0)} ({anomaly_counts.get(True, 0)/len(df)*100:.1f}%)')

## 3. Temporal Analysis

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.day_name()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hour distribution
df['hour'].hist(bins=24, ax=axes[0], edgecolor='black', alpha=0.7)
axes[0].set_title('Access Events by Hour of Day')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Count')
axes[0].axvline(x=9, color='r', linestyle='--', label='Work hours start')
axes[0].axvline(x=17, color='r', linestyle='--', label='Work hours end')
axes[0].legend()

# Day of week distribution
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['day_of_week'].value_counts().reindex(day_order).plot(kind='bar', ax=axes[1], edgecolor='black', alpha=0.7)
axes[1].set_title('Access Events by Day of Week')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Anomaly Analysis

In [ ]:
if 'is_anomaly' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Anomaly by hour
    pd.crosstab(df['hour'], df['is_anomaly']).plot(kind='bar', stacked=True, ax=axes[0], alpha=0.7)
    axes[0].set_title('Normal vs Anomalous Events by Hour')
    axes[0].set_xlabel('Hour')
    axes[0].set_ylabel('Count')
    axes[0].legend(['Normal', 'Anomaly'])

    # Anomaly types
    if 'anomaly_type' in df.columns:
        anomaly_types = df[df['is_anomaly'] == True]['anomaly_type'].value_counts()
        anomaly_types.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
        axes[1].set_title('Anomaly Type Distribution')
        axes[1].set_ylabel('')

    plt.tight_layout()
    plt.show()

## 5. Resource & Action Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Resource distribution
df['resource'].value_counts().head(10).plot(kind='barh', ax=axes[0], alpha=0.7)
axes[0].set_title('Top 10 Accessed Resources')
axes[0].set_xlabel('Count')

# Action distribution
df['action'].value_counts().plot(kind='bar', ax=axes[1], alpha=0.7, edgecolor='black')
axes[1].set_title('Action Distribution')
axes[1].set_xlabel('Action')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 6. Success Rate Analysis

In [ ]:
success_rate = df['success'].mean()
print(f'Overall success rate: {success_rate:.2%}')

if 'is_anomaly' in df.columns:
    normal_success = df[df['is_anomaly'] == False]['success'].mean()
    anomaly_success = df[df['is_anomaly'] == True]['success'].mean()
    print(f'Normal event success rate: {normal_success:.2%}')
    print(f'Anomalous event success rate: {anomaly_success:.2%}')

## 7. Key Insights

1. **Temporal Patterns**: Most access events occur during working hours (9 AM - 5 PM)
2. **Weekday Bias**: Significantly more events on weekdays vs weekends
3. **Anomaly Distribution**: ~10% of events are labeled as anomalies
4. **Common Resources**: Dashboard, documents, and settings are most accessed
5. **Read-Heavy**: Most actions are read operations

These patterns will be captured by our 15-dimensional feature vector for the anomaly detection model.